# **Day 4 — Attention & Transformers**

### **Attention**

As we discussed in Day 3, one of RNN's limitation is that it does not feature parallelism and the Vanishing Gradient Problem. Attention comes handy in solving this issue. Instead of passing information sequentially, attention identifies elements in a sequence that are associated with each other. Instead of compressing a sequence into a hidden state, **attention lets every position look directly at every other position and check how much each two are related**.

**For example** : *"The dog enjoys playing with its owner."*

Through attention, "it" refers to the noun it is associated with.  It assigns different weights to different input elements enabling the model to prioritize certain information over others. This is **useful in textual classification tasks** - language translation, spam detection, sentiment recognition, etc.


**Self-attention vs. Multi-head attention:**
*self attention*: queries, keys and values *come from the same sequence*. *Multi_head attention*: runs *several attention computations in parallel* and their outputs get combined.

### **Transformers**

a transformer is a type of neural network architecture designed to **handle sequential data** primarily for tasks such as language translation, text generation and many more. Unlike traditional recurrent neural networks (RNNs) or convolutional neural networks (CNNs), *Transformers uses attention mechanism to capture relationships between all words in a sentence* regardless of their distance from each other.

*For today's tasks we will use the DistilBERT pre-trained transformer.*



## **Imports**

In [1]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score
import numpy as np
from transformers import pipeline, DataCollatorWithPadding

## **Uploading the Dataset**

In [2]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')

train_path="/content/drive/MyDrive/train.csv"
test_path="/content/drive/MyDrive/test.csv"

train_df = pd.read_csv(train_path, header=None, names=["label", "title", "description"])
test_df = pd.read_csv(test_path, header=None, names=["label", "title", "description"])

print("Extraction complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extraction complete!


*The dataset is a news classification datset that sorts news into 4 distinct categories - World, Sports, Business and Tech/Sci.*

## **Pre-trained Pipeline**

In [3]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sample_text = "The central bank raised interest rates again this week amid inflation concerns."
labels=["World", "sports", "Business", "Sci/Tech"]



Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [17]:
result = classifier(sample_text, candidate_labels=labels)
print(result["labels"][0], "--->", round(result["scores"][0], 3))

World ---> 0.454


For the sample text - *"The central bank raised interest rates again this week amid inflation concerns."* - the model predixted the news category as "World" with a 45.4% confidence which is faitly low. This indicates that **the model was genuinly uncertain, not confidently wrong**.

## **Fine-tuned DistilBERT Model**

In [5]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (120001, 3)
Test shape: (7601, 3)


In [6]:
train_df["label"] = pd.to_numeric(train_df["label"], errors="coerce")
test_df["label"] = pd.to_numeric(test_df["label"], errors="coerce")


train_df = train_df.dropna(subset=["label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["label"]).reset_index(drop=True)

In [7]:
print(train_df["label"].unique())
print(test_df["label"].unique())

[3. 4. 2. 1.]
[3. 4. 2. 1.]


In [8]:
train_df.columns

Index(['label', 'title', 'description'], dtype='object')

In [9]:
train_df["text"] = train_df["title"] + " " + train_df["description"]
test_df["text"] = test_df["title"] + " " + test_df["description"]

train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"] = test_df["label"].astype(int) - 1

train_ds = Dataset.from_pandas(train_df[["text", "label"]])
test_ds = Dataset.from_pandas(test_df[["text", "label"]])

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
  return tokenizer(batch["text"], max_length=128, truncation=True)

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=1)
  return {
      "accuracy": accuracy_score(labels, preds),
      "f1_macro": f1_score(labels, preds, average="macro")
      }

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ag_news_distilbert",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()
transformer_metrics = trainer.evaluate()
print("Transformer (DistilBERT)", transformer_metrics)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.559586
200,0.391721
300,0.371955
400,0.316710
500,0.316321
600,0.299307
700,0.290013
800,0.263302
900,0.298104
1000,0.254395


Training Loss,Validation Loss,Step,Accuracy,F1 Macro
0.189093,0.173630,7500,0.945526,0.945524


Transformer (DistilBERT) {'eval_loss': 0.1736300140619278, 'eval_accuracy': 0.9455263157894737, 'eval_f1_macro': 0.9455235813526355}


In [14]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


MAX_WORDS = 20000
MAX_LEN = 128

tok = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tok.fit_on_texts(train_df["text"])

X_train = pad_sequences(tok.texts_to_sequences(train_df["text"]), maxlen=MAX_LEN)
X_test = pad_sequences(tok.texts_to_sequences(test_df["text"]), maxlen=MAX_LEN)
y_train = train_df["label"].values
y_test = test_df["label"].values

lstm_model = Sequential([
    Embedding(MAX_WORDS, 128, input_length=MAX_LEN),
    LSTM(64),
    Dense(4, activation="softmax"),
])

lstm_model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
lstm_model.fit(X_train, y_train, validation_split=0.1, batch_size=64, epochs=3)

lstm_loss, lstm_accuracy = lstm_model.evaluate(X_test, y_test)
print("LSTM test Accuracy:", lstm_accuracy)

Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - accuracy: 0.8847 - loss: 0.3408 - val_accuracy: 0.9092 - val_loss: 0.2629
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 18s 11ms/step - accuracy: 0.9352 - loss: 0.1993 - val_accuracy: 0.9103 - val_loss: 0.2605
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.9494 - loss: 0.1508 - val_accuracy: 0.9065 - val_loss: 0.2812
238/238 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9157 - loss: 0.2655
LSTM test Accuracy: 0.9156578779220581


## **Transformer vs. LSTM from Day 3**

| Model | Accuracy | Information |
|---|---|---|
| **LSTM** | 91.5% | 3 epochs; validation accuracy peaked at epoch 2 then returned to initial value |
| **DistilBERT fine-tuned transformer** | 94.5% | 1 epoch ; F1 macro of 0.94, no overfitting signs |

DistilBERT outperformed LSTM by 3 accuracy points while training for fewer epochs and showing no degradation. This supports the idea that parallelism has importance in processing long sequential data.

# **Attention vs. RNN**

An **RNN** processes a sequence one timestamp at a time *italicized text* compressing everything it has processed so far into a singlr fixed-size hidden state that carries forward;* information from very early parts of a long sequence have to survive many sequential updates* before it can influence later prediction which often gets overwritten along the way.

**Attention**, by contrast, lets the model look directly at every position regardless of distance; a word at the very beginning of a sentance can equally influence a prediction as a word at the very end. This was makes attention **far more parallelizable**  during training time although it comes at the cost of consuming the processor's resources as it *compares every position to every other position*.

## **Recorded Architecture Decision**

**Core model decision:**

DistilBERT - transformer - will serve as this project's core model.

**Reasoning:**

DistilBERT reached a **94.5% test accuracy and F1 macro of 0.945** versus *91.5% for the Day 3 LSTM* baseline on the same AG news dataset. The 3 point gap was achieved with fewer training epochs with no overfitting. Beyond the raw metric gap, **self-attention lets the model weigh every word**  in a headline or description against every other word directly, rather than compressing the sequence step by step into one hidden state - *this maters for news text where the topic defining word isn't near the start of the sentence*. Given its stronger accuracy, faster convergence  and practical training time - around 6.5 mins  on a single Cloab T4 GPU - **DistiBERT is the clear choice for this project's core model**.